# Notebook 2: Baseline Evaluation — NeMo Evaluator + LLM-as-a-Judge
---

**Objective**: Rigorously evaluate the baseline model responses using both automated metrics (Exact Match, F1) and LLM-as-a-Judge (correctness, faithfulness, conciseness). This follows the NeMo Evaluator pattern from the DLI course.

**DLI Course Mapping**: Learning Objective 2 — "Evaluate with NeMo Evaluator + LLM-as-a-Judge"

**What you'll do**:
1. Load baseline responses from Notebook 1
2. Compute automated metrics (Exact Match, F1 — GSM8K-style)
3. Run LLM-as-a-Judge evaluation (correctness, faithfulness, conciseness)
4. Log all metrics to MLflow
5. Establish the baseline benchmark to beat

In [ ]:
# ============================================================
# Setup & Imports
# ============================================================
import os
import sys
import json
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from utils import (
    NIMInferenceClient,
    LLMJudge,
    set_seed,
    save_results,
    load_results,
    compute_exact_match,
    compute_f1_score,
    print_metrics_summary,
    log_metrics_to_mlflow,
    plot_comparison_bar_chart,
    RESULTS_DIR,
)

set_seed(42)

## 1. Load Baseline Responses

In [ ]:
# ============================================================
# Load baseline data from Notebook 1
# ============================================================

baseline_df = load_results("baseline_responses.csv")
test_df = load_results("test_split.csv")

print(f"Loaded {len(baseline_df)} baseline responses")
print(f"Columns: {list(baseline_df.columns)}")
print(f"\nSample response: {baseline_df.iloc[0]['model_response'][:200]}")

## 2. Automated Metrics — GSM8K-Style

Following the DLI course, we compute standard QA metrics before running the judge.

> **From DLI Course**: Always start with cheap automated metrics to get a quick signal before investing in expensive LLM-based evaluation.

In [ ]:
# ============================================================
# Compute Automated Metrics — From NVIDIA DLI Course Lab
# ============================================================

predictions = baseline_df["model_response"].tolist()
references = baseline_df["answer"].tolist()

# Exact Match
em_score = compute_exact_match(predictions, references)

# Token-level F1 (GSM8K-style)
f1_score = compute_f1_score(predictions, references)

# Response length analysis
avg_pred_len = np.mean([len(p.split()) for p in predictions])
avg_ref_len = np.mean([len(r.split()) for r in references])

baseline_auto_metrics = {
    "exact_match": em_score,
    "f1_score": f1_score,
    "avg_prediction_length_words": avg_pred_len,
    "avg_reference_length_words": avg_ref_len,
    "length_ratio": avg_pred_len / max(avg_ref_len, 1),
}

print_metrics_summary(baseline_auto_metrics, "Base Llama-3.1-8B — Automated Metrics")

## 3. LLM-as-a-Judge Evaluation

This is the core of the NeMo Evaluator workflow. We use a larger model (Llama-3.1-70B) as a judge to evaluate three dimensions:

1. **Correctness**: Does the answer match the ground truth?
2. **Faithfulness**: Is the answer grounded in the provided evidence?
3. **Conciseness**: Is the answer appropriately brief?

> **From DLI Course**: The judge prompt templates follow the exact NeMo Evaluator custom judge format — structured prompts that return JSON with score + reasoning.

In [ ]:
# ============================================================
# LLM-as-a-Judge Setup — From NVIDIA DLI Course Lab
# ============================================================

# Initialize NIM client (will be used for judge queries)
nim_client = NIMInferenceClient(model="meta/llama-3.1-8b-instruct")

# Initialize the judge (uses a larger model for evaluation)
judge = LLMJudge(
    client=nim_client,
    judge_model="meta/llama-3.1-70b-instruct",  # Larger model = better judge
)

print("LLM-as-a-Judge initialized")
print(f"Judge model: meta/llama-3.1-70b-instruct")
print(f"Criteria: correctness, faithfulness, conciseness")

In [ ]:
# ============================================================
# Run Judge Evaluation on Baseline — From NVIDIA DLI Course Lab
# ============================================================

print("Running LLM-as-a-Judge evaluation...")
print(f"Evaluating {len(baseline_df)} examples across 3 criteria...")
print("(This may take several minutes due to API rate limits)\n")

judge_results = judge.evaluate_batch(
    questions=baseline_df["question"].tolist(),
    predictions=baseline_df["model_response"].tolist(),
    references=baseline_df["answer"].tolist(),
    evidences=baseline_df.get("evidence", pd.Series([""] * len(baseline_df))).tolist(),
    criteria=["correctness", "faithfulness", "conciseness"],
)

print(f"\nJudge evaluation complete: {len(judge_results)} examples evaluated")
print(f"Columns: {list(judge_results.columns)}")

In [ ]:
# ============================================================
# Aggregate Judge Scores
# ============================================================

# Compute averages for each criterion
judge_metrics = {}
for criterion in ["correctness", "faithfulness", "conciseness"]:
    col = f"{criterion}_score"
    if col in judge_results.columns:
        scores = judge_results[col].dropna()
        judge_metrics[f"judge_{criterion}_mean"] = scores.mean()
        judge_metrics[f"judge_{criterion}_std"] = scores.std()
        judge_metrics[f"judge_{criterion}_median"] = scores.median()

# Overall judge score
score_cols = [c for c in judge_results.columns if c.endswith("_score")]
if score_cols:
    judge_results["avg_judge_score"] = judge_results[score_cols].mean(axis=1)
    judge_metrics["judge_avg_score"] = judge_results["avg_judge_score"].mean()

print_metrics_summary(judge_metrics, "Base Llama-3.1-8B — Judge Metrics")

## 4. Combined Baseline Metrics & MLflow Logging

In [ ]:
# ============================================================
# Combine All Baseline Metrics
# ============================================================

all_baseline_metrics = {**baseline_auto_metrics, **judge_metrics}
print_metrics_summary(all_baseline_metrics, "Base Llama-3.1-8B — ALL METRICS")

# Save detailed results
save_results(judge_results, "baseline_judge_results.csv")
save_results(all_baseline_metrics, "baseline_metrics.json")

In [ ]:
# ============================================================
# Log to MLflow — From NVIDIA DLI Course Lab
# ============================================================

log_metrics_to_mlflow(
    metrics=all_baseline_metrics,
    run_name="baseline_llama31_8b",
    experiment_name="financebench-llm",
    params={
        "model": "meta/llama-3.1-8b-instruct",
        "method": "baseline",
        "dataset": "PatronusAI/financebench",
        "test_size": len(baseline_df),
        "temperature": 0.1,
        "max_tokens": 512,
    },
)

print("Metrics logged to MLflow successfully.")

## 5. Visualize Baseline Results

In [ ]:
# ============================================================
# Baseline Score Distribution — From NVIDIA DLI Course Lab
# ============================================================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, criterion in enumerate(["correctness", "faithfulness", "conciseness"]):
    col = f"{criterion}_score"
    if col in judge_results.columns:
        ax = axes[i]
        scores = judge_results[col].dropna()
        ax.hist(scores, bins=[0.5, 1.5, 2.5, 3.5, 4.5, 5.5],
                color="#667eea", edgecolor="black", alpha=0.8)
        ax.set_title(f"{criterion.title()} (mean={scores.mean():.2f})", fontsize=12)
        ax.set_xlabel("Score (1-5)")
        ax.set_ylabel("Count")
        ax.axvline(scores.mean(), color="red", linestyle="--", label=f"Mean: {scores.mean():.2f}")
        ax.legend()

plt.suptitle("Baseline LLM-as-a-Judge Score Distributions", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / "baseline_judge_distributions.png"), dpi=150, bbox_inches="tight")
plt.close()
print("Chart saved to results/baseline_judge_distributions.png")

In [ ]:
# ============================================================
# Summary & Next Steps
# ============================================================

print("=" * 60)
print("  BASELINE EVALUATION COMPLETE")
print("=" * 60)
print(f"\n  Exact Match:   {baseline_auto_metrics['exact_match']:.4f}")
print(f"  F1 Score:      {baseline_auto_metrics['f1_score']:.4f}")
print(f"  Correctness:   {judge_metrics.get('judge_correctness_mean', 0):.2f} / 5")
print(f"  Faithfulness:  {judge_metrics.get('judge_faithfulness_mean', 0):.2f} / 5")
print(f"  Conciseness:   {judge_metrics.get('judge_conciseness_mean', 0):.2f} / 5")
print(f"\n  These are our targets to beat with ICL (Notebook 3)")
print(f"  and LoRA fine-tuning (Notebook 4).")
print("\n✓ Notebook 2 complete — proceed to Notebook 3 for ICL.")